In [1]:
from pyspark.sql.functions import when
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CDC").getOrCreate()

In [2]:
df = spark.read.csv(r"C:\Users\Mps\Downloads\OFFICEDATA")


df.show(truncate = False)

+---+------------------+----------------+
|_c0|_c1               |_c2             |
+---+------------------+----------------+
|0  |'Herman Zimmerman'|'Oklahoma City';|
|1  |'Lisa Ray'        |'Columbus';     |
|2  |'Terrell Reeves'  |'Jacksonville'; |
|3  |'Steve Goodwin'   |'Charlotte';    |
|4  |'Leah Tran'       |'Detroit';      |
|5  |'Wilbert Holmes'  |'Washington';   |
|6  |'Mindy George'    |'Los Angeles';  |
|7  |'Rosa Huff'       |'Phoenix';      |
|8  |'Clayton Jennings'|'Denver';       |
|9  |'Darla Hayes'     |'Charlotte';    |
|10 |'Jack Hicks'      |'Houston';      |
|11 |'Francis Davidson'|'Austin';       |
|12 |'Jerome Padilla'  |'San Francisco';|
|13 |'Mamie Duncan'    |'Houston';      |
|14 |'Julia Cain'      |'San Jose';     |
|15 |'Leslie Klein'    |'Seattle';      |
|16 |'Isaac Bridges'   |'Philadelphia'; |
|17 |'Martin Adkins'   |'Chicago';      |
|18 |'Vincent Perry'   |'Detroit';      |
|19 |'William Porter'  |'Fort Worth';   |
+---+------------------+----------

In [2]:
from pyspark.sql.functions import when
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("pyspark project") \
    .config(
        "spark.driver",
        r"C:\Users\Mps\Downloads\mysql-connector-j-8.0.33\mysql-connector-j-8.0.33\mysql-connector-j-8.0.33.jar"
    ) .getOrCreate()

full_load = spark.read.csv(
    r"C:\Users\Mps\Downloads\OFFICEDATA",
    header=False,
    inferSchema=True
)

full_load = full_load \
    .withColumnRenamed("_c0", "id") \
    .withColumnRenamed("_c1", "fullname") \
    .withColumnRenamed("_c2", "city")

full_load.show()
Updated_load = spark.read.csv(
    r"C:\Users\Mps\Downloads\officeD",
    header=False,
    inferSchema=True
)

Updated_load = Updated_load \
    .withColumnRenamed("_c0", "status") \
    .withColumnRenamed("_c1", "id") \
    .withColumnRenamed("_c2", "fullname") \
    .withColumnRenamed("_c3", "city")
Updated_load.show()
for row in Updated_load.collect():
    print(row)

    if row["status"] == "U":
        full_load = full_load.withColumn("fullname",when(full_load["id"] == row["id"],row["fullname"]).otherwise(full_load["fullname"]))
        full_load = full_load.withColumn("city",when(full_load["id"] == row["id"],row["city"]).otherwise(full_load["city"]))
    if row["status"] == "I":
        insertedRow = [list(row)[1:]]
        columns = ["id","FullName","city"]
        newDF = spark.createDataFrame(insertedRow,columns)
        full_load = full_load.union(newDF)

    if row["status"] == "D":
        full_load =full_load.filter(full_load.id !=row["id"])
full_load.show(120)
full_load.write \
    .format("jdbc") \
    .option("url", "jdbc:mysql://localhost:3306/ahmad_schema") \
    .option("dbtable", "ahmad_schema.Persons") \
    .option("user", "root") \
    .option("password", "rootroot") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .mode("overwrite") \
    .save()

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it